#1. Análisis Exploratorio y de Negocio: Brazilian E-Commerce (Olist)

## Datos Académicos
* **Materia:** Ciencia de Datos
* **Institución:** Instituto Tecnológico Superior de Alvarado (ITSAV)
* **Profesor:** I.S.C. Fernando Mayorga Guittins
* **Integrantes del Equipo A:**
  1. Hiromi Alejandra Mosqueda Salto (Líder Técnico)
  2. Christopher Vasquez Carvajal (Desarrollador - Limpieza)
  3. Erik Alberto Hernández Orozco(Desarrollador - Ingeniería de Datos)
  4. Alberto Quevedo Gapi (Desarrollador - EDA y Visualización)
* **Fecha:** Octubre 2026

## Contexto de los Datos
El conjunto de datos analizado pertenece a **Olist**, la plataforma de comercio electrónico más grande de Brasil. Contiene información real y anonimizada de aproximadamente **100,000 órdenes de compra** realizadas entre 2016 y 2018. El modelo relacional abarca múltiples dimensiones comerciales: estado del pedido, atributos físicos del producto, pagos, opiniones de clientes y tiempos logísticos de entrega.

## Objetivos y Preguntas de Negocio
El objetivo principal es transformar datos crudos en métricas accionables respondiendo a tres interrogantes estratégicas:
1. **Pregunta 1 (Satisfacción Logística):** ¿Cómo impacta el tiempo de entrega real frente al estimado en la calificación de satisfacción del cliente? *(Cruce de `olist_orders` con `olist_order_reviews`)*.
2. **Pregunta 2 (Rendimiento Comercial):** ¿Cuáles son las 5 categorías de productos que generan mayores ingresos económicos en la plataforma? *(Cruce de `olist_order_items`, `olist_products` y `product_category_name_translation`)*.
3. **Pregunta 3 (Análisis Financiero):** ¿Cuál es la relación entre el método de pago elegido por el usuario y el valor total del ticket de compra? *(Cruce de `olist_orders` con `olist_order_payments`)*.


##2. Importación de Librerías y Configuración
*(Responsable: Hiromi Alejandra Mosqueda Salto)*

In [23]:
# =====================================================================
# SECCIÓN 2: IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN GLOBAL
# =====================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# Configuración global de Pandas para visualización de DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Configuración de estilo gráfico para Seaborn y Matplotlib
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

# Silenciar advertencias de ejecución
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías e importaciones configuradas exitosamente.")


✅ Librerías e importaciones configuradas exitosamente.


## 3. Carga e Inspección Inicial de los Datos

En esta sección se realiza el montaje de Google Drive para acceder a la carpeta `Proyecto_Final/data/` y se cargan los 6 archivos CSV que conforman el ecosistema de Olist.


In [24]:
# =====================================================================
# SECCIÓN 3: MONTAJE Y CARGA DE DATANETS DE OLIST
# =====================================================================

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir ruta absoluta hacia la carpeta de datos
DATA_PATH = '/content/drive/MyDrive/Proyecto_Final/data/'

# 3. Cargar las 6 tablas principales del proyecto
df_orders = pd.read_csv(os.path.join(DATA_PATH, 'olist_orders_dataset.csv'))
df_order_items = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_items_dataset.csv'))
df_products = pd.read_csv(os.path.join(DATA_PATH, 'olist_products_dataset.csv'))
df_reviews = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_reviews_dataset.csv'))
df_payments = pd.read_csv(os.path.join(DATA_PATH, 'olist_order_payments_dataset.csv'))
df_translation = pd.read_csv(os.path.join(DATA_PATH, 'product_category_name_translation.csv'))

print("✅ Todos los datasets fueron cargados correctamente.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Todos los datasets fueron cargados correctamente.


In [25]:
# =====================================================================
# INSPECCIÓN ESTRUCTURAL INICIAL DE LAS TABLAS PRINCIPALES
# =====================================================================

print("=== 1. FORMA DE LOS DATASETS (Filas, Columnas) ===")
print(f"Orders:   {df_orders.shape}")
print(f"Items:    {df_order_items.shape}")
print(f"Products: {df_products.shape}")
print(f"Reviews:  {df_reviews.shape}")
print(f"Payments: {df_payments.shape}")

print("\n=== 2. VISTA PREVIA DE PEDIDOS (df_orders.head()) ===")
display(df_orders.head(3))

print("\n=== 3. INFORMACIÓN DE TIPOS Y NULOS EN PEDIDOS (df_orders.info()) ===")
df_orders.info()

print("\n=== 4. RESUMEN ESTADÍSTICO DE PAGOS (df_payments.describe()) ===")
display(df_payments.describe())


=== 1. FORMA DE LOS DATASETS (Filas, Columnas) ===
Orders:   (99441, 8)
Items:    (112650, 7)
Products: (32951, 9)
Reviews:  (99224, 7)
Payments: (103886, 5)

=== 2. VISTA PREVIA DE PEDIDOS (df_orders.head()) ===


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



=== 3. INFORMACIÓN DE TIPOS Y NULOS EN PEDIDOS (df_orders.info()) ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

=== 4. RESUMEN ESTADÍSTICO DE PAGOS (df_payments.describe()) ===


,payment_sequential,payment_installments,payment_value
count,103886.00,103886.00,103886.00
mean,1.09,2.85,154.10
std,0.71,2.69,217.49
min,1.00,0.00,0.00
25%,1.00,1.00,56.79
50%,1.00,1.00,100.00
75%,1.00,4.00,171.84
max,29.00,24.00,13664.08


## 4. Limpieza y Preprocesamiento
*(Responsable: Christopher Vasquez Carvajal )*

En esta sección se aborda la calidad de los datos de las 6 tablas de Olist. Se identifican y tratan valores faltantes, se eliminan registros duplicados, se realiza el casteo explícito de tipos de datos (especialmente cadenas de texto a formato fecha `datetime`) y se estandarizan las variables categóricas.


### 4.1 Tratamiento de Valores Nulos

**Análisis y Estrategia:**
1. **`df_orders`:** Las columnas de fecha de entrega (`order_delivered_customer_date`) presentan valores nulos cuando el pedido no ha sido entregado o fue cancelado. Para nuestro análisis logístico (Pregunta 1), nos enfocaremos únicamente en los pedidos con estado `'delivered'`, por lo que se descartarán los registros sin fecha de entrega efectiva.
2. **`df_products`:** La columna `product_category_name` tiene valores vacíos. Imputaremos la etiqueta `'outro'` (otro) para evitar perder el producto al cruzar con la tabla de traducción de categorías.
3. **`df_reviews`:** Las columnas `review_comment_title` y `review_comment_message` presentan un alto porcentaje de nulos debido a que muchos clientes solo asignan la calificación en estrellas sin escribir texto. Se imputarán con la cadena `'Sin comentario'`.


In [26]:
# =====================================================================
# 4.1 TRATAMIENTO DE VALORES NULOS
# =====================================================================

print("=== NULOS INICIALES EN ORDERS ===")
print(df_orders.isnull().sum()[df_orders.isnull().sum() > 0])

# 1. Filtrar solo pedidos entregados para análisis logístico preciso
df_orders = df_orders[df_orders['order_status'] == 'delivered'].copy()
df_orders.dropna(subset=['order_delivered_customer_date', 'order_approved_at'], inplace=True)

# 2. Imputación en la tabla de Productos
df_products['product_category_name'].fillna('outro', inplace=True)

# 3. Imputación en la tabla de Reseñas / Reviews
df_reviews['review_comment_title'].fillna('Sin comentario', inplace=True)
df_reviews['review_comment_message'].fillna('Sin comentario', inplace=True)

print("\n✅ Tratamiento de nulos finalizado.")
print(f"Pedidos entregados válidos: {len(df_orders):,}")

=== NULOS INICIALES EN ORDERS ===
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

✅ Tratamiento de nulos finalizado.
Pedidos entregados válidos: 96,456


### 4.2 Manejo de Duplicados
Se verifica la presencia de registros totalmente idénticos o duplicados en las claves primarias (`order_id`, `review_id`, `product_id`). En el caso de las reseñas (`df_reviews`), cuando existen múltiples encuestas enviadas para un mismo pedido, se conserva únicamente la última respuesta registrada.


In [27]:
# =====================================================================
# 4.2 MANEJO DE DUPLICADOS
# =====================================================================

print(f"Duplicados en Orders: {df_orders.duplicated().sum()}")
print(f"Duplicados en Items: {df_order_items.duplicated().sum()}")
print(f"Duplicados en Reviews: {df_reviews.duplicated(subset=['review_id']).sum()}")

# Eliminar duplicados en Reviews conservando el último registro
df_reviews.drop_duplicates(subset=['review_id'], keep='last', inplace=True)

# Eliminar cualquier fila totalmente duplicada por seguridad
df_orders.drop_duplicates(inplace=True)
df_order_items.drop_duplicates(inplace=True)
df_products.drop_duplicates(inplace=True)

print("✅ Eliminación de duplicados completada.")

Duplicados en Orders: 0
Duplicados en Items: 0
Duplicados en Reviews: 814
✅ Eliminación de duplicados completada.


### 4.3 Corrección de Tipos de Datos (Casteo)
Las columnas con información temporal en `df_orders` y `df_reviews` fueron importadas como cadenas de texto (`object`). Se realiza la conversión explícita a tipo `datetime64[ns]` mediante `pd.to_datetime()` para habilitar operaciones aritméticas con fechas y calcular tiempos reales de entrega.


In [28]:
# =====================================================================
# 4.3 CASTEO DE COLUMNAS A DATETIME
# =====================================================================

columnas_fecha_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

# Convertir columnas de fecha en df_orders
for col in columnas_fecha_orders:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')

# Convertir columnas de fecha en df_reviews
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'])
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'])

print("=== VERIFICACIÓN DE TIPOS EN ORDERS ===")
print(df_orders[columnas_fecha_orders].dtypes)

=== VERIFICACIÓN DE TIPOS EN ORDERS ===
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


### 4.4 Estandarización de Texto
Se aplica normalización a las columnas de texto categóricas (`order_status`, `payment_type`, `product_category_name`): conversión a minúsculas y eliminación de espacios en blanco al inicio o final (`strip()`).

In [29]:
# =====================================================================
# 4.4 ESTANDARIZACIÓN DE VARIABLES CATEGÓRICAS
# =====================================================================

df_orders['order_status'] = df_orders['order_status'].astype(str).str.strip().str.lower()
df_payments['payment_type'] = df_payments['payment_type'].astype(str).str.strip().str.lower()
df_products['product_category_name'] = df_products['product_category_name'].astype(str).str.strip().str.lower()
df_translation['product_category_name'] = df_translation['product_category_name'].astype(str).str.strip().str.lower()
df_translation['product_category_name_english'] = df_translation['product_category_name_english'].astype(str).str.strip().str.lower()

print("✅ Estandarización de textos finalizada con éxito.")


✅ Estandarización de textos finalizada con éxito.


## 5. Enriquecimiento e Ingeniería de Características
*(Responsable: Erik Alberto Hernández Orozco)*

En esta sección realizamos la integración relacional de los datasets mediante `pd.merge()` y la creación de nuevas variables derivadas. Aplicamos **operaciones vectorizadas de NumPy** (`np.where`, `np.select`) para evitar bucles lentos y realizamos la detección estadística de valores atípicos (*outliers*) en variables financieras y logísticas.


### 5.1 Unión de Datos (Merge/Join)

Para dar respuesta a las 3 preguntas de negocio planteadas en la Fase 1, enriquecemos la tabla principal `df_orders` realizando tres uniones de tipo modelo relacional:
1. **`df_orders` + `df_reviews`:** Relaciona el estado logístico del pedido con la calificación otorgada por el cliente (`review_score`).
2. **`df_order_items` + `df_products` + `df_translation`:** Vincula los productos vendidos con sus precios y la traducción al inglés de sus categorías.
3. **`df_orders` + `df_payments`:** Asocia cada orden de compra con el método de pago utilizado y el valor total de la transacción.


In [30]:
# =====================================================================
# 5.1 UNIÓN DE DATASETS MEDIANTE PD.MERGE()
# =====================================================================

# 1. Cruce para Pregunta 1: Pedidos con Reseñas de Clientes
df_orders_reviews = pd.merge(
    df_orders,
    df_reviews[['order_id', 'review_score', 'review_comment_message']],
    on='order_id',
    how='inner'
)

# 2. Cruce para Pregunta 2: Ítems de Orden con Productos y Traducción de Categorías
df_items_products = pd.merge(
    df_order_items[['order_id', 'product_id', 'price', 'freight_value']],
    df_products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

df_items_products = pd.merge(
    df_items_products,
    df_translation,
    on='product_category_name',
    how='left'
)

# Rellenar categorías sin traducción con 'desconocido'
df_items_products['product_category_name_english'].fillna('desconocido', inplace=True)

# 3. Cruce para Pregunta 3: Pedidos con Métodos de Pago
df_orders_payments = pd.merge(
    df_orders[['order_id', 'customer_id', 'order_status']],
    df_payments[['order_id', 'payment_type', 'payment_value', 'payment_installments']],
    on='order_id',
    how='inner'
)

print("=== VERIFICACIÓN DE TAMAÑOS TRAS CRUCES ===")
print(f"Orders + Reviews:  {df_orders_reviews.shape[0]:,} filas")
print(f"Items + Products:  {df_items_products.shape[0]:,} filas")
print(f"Orders + Payments: {df_orders_payments.shape[0]:,} filas")

=== VERIFICACIÓN DE TAMAÑOS TRAS CRUCES ===
Orders + Reviews:  95,581 filas
Items + Products:  112,650 filas
Orders + Payments: 100,734 filas


### 5.2 Operaciones Vectorizadas con NumPy

En lugar de utilizar iteraciones lentas con bucles `for`, empleamos funciones vectorizadas de NumPy para calcular métricas de tiempo e imponer reglas lógicas de clasificación masiva:
* **`dias_diferencia_entrega`:** Diferencia en días entre la fecha estimada de entrega y la fecha real de recepción del pedido.
* **`estado_entrega` (`np.select`):** Clasificación lógica del desempeño logístico en tres niveles: `'Anticipado (>3 días)'`, `'A tiempo (0 a 3 días)'` y `'Con retraso (<0 días)'`.
* **`nivel_satisfaccion` (`np.where`):** Indicador binario que clasifica el puntaje del cliente como `'Satisfecho'` (4-5 estrellas) o `'Insatisfecho'` (1-3 estrellas).


In [31]:
# =====================================================================
# 5.2 INGENIERÍA DE CARACTERÍSTICAS CON OPERACIONES VECTORIZADAS
# =====================================================================

# 1. Cálculo vectorizado de diferencia de días entre estimado y real
# Un valor positivo indica que el pedido llegó antes de la fecha estimada
diferencia_segundos = (
    df_orders_reviews['order_estimated_delivery_date'] -
    df_orders_reviews['order_delivered_customer_date']
).dt.total_seconds()

df_orders_reviews['dias_diferencia_entrega'] = diferencia_segundos / (24 * 3600)

# 2. Clasificación lógica multicondicional usando np.select()
condiciones_entrega = [
    df_orders_reviews['dias_diferencia_entrega'] > 3,
    (df_orders_reviews['dias_diferencia_entrega'] >= 0) & (df_orders_reviews['dias_diferencia_entrega'] <= 3),
    df_orders_reviews['dias_diferencia_entrega'] < 0
]

opciones_entrega = ['Anticipado (>3 días)', 'A tiempo (0 a 3 días)', 'Con retraso (<0 días)']

df_orders_reviews['estado_entrega'] = np.select(condiciones_entrega, opciones_entrega, default='Desconocido')

# 3. Clasificación binaria con np.where() para satisfacción del cliente
df_orders_reviews['nivel_satisfaccion'] = np.where(
    df_orders_reviews['review_score'] >= 4,
    'Satisfecho',
    'Insatisfecho'
)

print("=== DISTRIBUCIÓN VECTORIZADA DEL ESTADO DE ENTREGA ===")
print(df_orders_reviews['estado_entrega'].value_counts())


=== DISTRIBUCIÓN VECTORIZADA DEL ESTADO DE ENTREGA ===
estado_entrega
Anticipado (>3 días)     83236
Con retraso (<0 días)     7643
A tiempo (0 a 3 días)     4702
Name: count, dtype: int64


### 5.3 Detección y Manejo de Valores Atípicos (Outliers)

Utilizamos funciones estadísticas de NumPy (`np.percentile`, `np.mean`, `np.std`) para analizar la dispersión de las variables numéricas continuas: el valor del pago (`payment_value`) y el precio de los productos (`price`).

Se implementa el criterio del **Rango Intercuartílico (IQR)** para identificar transacciones atípicas que podrían sesgar el análisis de promedios en los tickets de compra.


In [32]:
# =====================================================================
# 5.3 DETECCIÓN ESTADÍSTICA DE OUTLIERS CON NUMPY
# =====================================================================

pagos = df_orders_payments['payment_value'].values

# Cálculo de estadísticas descriptivas con NumPy
media_pago = np.mean(pagos)
std_pago = np.std(pagos)
q25 = np.percentile(pagos, 25)
q75 = np.percentile(pagos, 75)
iqr = q75 - q25

# Definición de límites para detección de outliers (IQR)
limite_superior_iqr = q75 + (1.5 * iqr)

# Identificación de observaciones atípicas
outliers_mask = pagos > limite_superior_iqr
cantidad_outliers = np.sum(outliers_mask)
porcentaje_outliers = (cantidad_outliers / len(pagos)) * 100

print("=== ANÁLISIS ESTADÍSTICO DE OUTLIERS EN PAGOS ===")
print(f"Media del Pago: ${media_pago:.2f} BRL | Desviación Estándar: ${std_pago:.2f} BRL")
print(f"Percentil 25 (Q1): ${q25:.2f} BRL | Percentil 75 (Q3): ${q75:.2f} BRL | IQR: ${iqr:.2f} BRL")
print(f"Límite Superior para Outliers: ${limite_superior_iqr:.2f} BRL")
print(f"Transacciones con valores atípicos superiores: {cantidad_outliers:,} ({porcentaje_outliers:.2f}%)")

# Para evitar distorsión en la visualización de la Pregunta 3, conservaremos un dataset filtrado sin atípicos extremos
df_orders_payments_clean = df_orders_payments[df_orders_payments['payment_value'] <= limite_superior_iqr].copy()
print(f"Dataset conservado para análisis sin atípicos extremos: {len(df_orders_payments_clean):,} filas")



=== ANÁLISIS ESTADÍSTICO DE OUTLIERS EN PAGOS ===
Media del Pago: $153.07 BRL | Desviación Estándar: $214.47 BRL
Percentil 25 (Q1): $56.78 BRL | Percentil 75 (Q3): $171.29 BRL | IQR: $114.51 BRL
Límite Superior para Outliers: $343.05 BRL
Transacciones con valores atípicos superiores: 7,695 (7.64%)
Dataset conservado para análisis sin atípicos extremos: 93,039 filas
